# temperature 와 재현성 — 같은 질문에 같은 답 받기

`1_1_openai` 에서 같은 질문을 두 번 보내 보면 **답이 매번 조금씩 다릅니다.**
이 노트북은 그 이유를 직접 확인하고, 답을 원하는 만큼 **고정하거나 풀어 주는** 방법을 다룹니다.

## 이 노트북의 흐름

| 단계 | 내용 |
|---|---|
| **Step 1~2** | 준비 — 키 확인, 호출을 짧게 해 줄 함수 만들기 |
| **Step 3** | 왜 답이 달라지나 — 모델이 단어를 **확률로** 고르는 모습 보기 |
| **Step 4~7** | temperature 실험 — 기본값 · 0 · 1.5 · `n` 으로 한 번에 여러 개 |
| **Step 8** | `seed` — "가능한 한" 같은 답을 달라고 부탁하기 |
| **Step 9** | 추론 모델(`gpt-5-nano`)은 temperature 를 못 바꾼다 |
| **Step 10** | 100% 같은 답 — 답을 저장해 두고 다시 쓰기 |

> **이 노트북에서 기억할 한 가지**
> temperature 는 모델이 다음 단어를 고를 때 **얼마나 모험을 할지** 정하는 값입니다.
> 0 으로 낮추면 거의 같은 답이 나오지만, API 로는 **100% 같은 답을 보장받을 수 없습니다.**

> **비용 안내**: 이 노트북은 API 를 15번 정도 호출합니다.
> 가장 싼 `gpt-4.1-nano` 를 쓰고 답 길이도 짧게 제한해 두었습니다.

## Step 1. 준비 — `1_1_openai` 와 똑같습니다

라이브러리를 불러오고, `.env` 에서 키를 읽고, 클라이언트를 만듭니다.
처음 보는 내용이 있다면 `1_1_openai` 의 Step 1~4 를 다시 보세요.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

openai = OpenAI()

## Step 2. 호출을 짧게 해 줄 함수

이번 노트북은 **같은 질문을 여러 번** 보냅니다.
매번 `messages` 를 만들고 `create()` 를 부르면 길어지므로 함수 하나로 묶습니다.

```python
def ask(question, **options):
```

`**options` 는 "그 밖의 설정은 받은 그대로 `create()` 에 넘겨라"라는 뜻입니다.
덕분에 `ask(질문, temperature=0)` 처럼 **실험할 값만 바꿔 가며** 부를 수 있습니다.

| 설정 | 값 | 이유 |
|---|---|---|
| `MODEL` | `gpt-4.1-nano` | temperature 를 지원하는 가장 싼 모델 |
| `max_tokens` | `60` | 답을 짧게 잘라 **비교하기 쉽고 요금도 적게** |

In [ ]:
MODEL = "gpt-4.1-nano"

def ask(question, **options):
    """질문 하나를 보내고 답변 글자만 돌려줍니다."""
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        max_tokens=60,
        **options,
    )
    return response.choices[0].message.content

## Step 3. 왜 답이 달라지나 — 모델은 단어를 확률로 고른다

모델은 문장을 한 번에 쓰지 않습니다. **다음에 올 토큰(단어 조각) 하나를 고르고, 또 하나를 고르는** 일을 반복합니다.
그때 후보가 하나가 아니라 **여러 개이고, 각각 확률이 붙어 있습니다.**

```python
logprobs=True, top_logprobs=5
```

이 두 옵션을 주면 모델이 **실제로 고민한 후보 5개와 확률**을 함께 돌려줍니다.
`logprob` 은 확률에 로그를 씌운 값이라 `math.exp()` 로 다시 확률로 바꿔 봅니다.

```
"보안 사고를 한 단어로 표현하면?" 의 첫 단어 후보 (모양을 보여 주는 예시)

  침해      62.1%   <- 가장 유력하지만
  위협      21.4%   <- 이것이 뽑힐 수도 있고
  해킹       9.8%   <- 이것이 뽑힐 수도 있다
```

**확률이 가장 높은 후보만 늘 뽑히는 게 아닙니다.** 그래서 같은 질문에도 답이 달라집니다.

In [ ]:
import math

response = openai.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "보안 사고를 한 단어로 표현하면? 단어 하나만 답하세요."}],
    max_tokens=5,
    logprobs=True,
    top_logprobs=5,
)

first_token = response.choices[0].logprobs.content[0]   # 답변의 첫 번째 토큰
print("실제로 뽑힌 토큰:", repr(first_token.token))
print()
print("모델이 고민한 후보들:")
for candidate in first_token.top_logprobs:
    print(f"  {candidate.token!r:12} {math.exp(candidate.logprob) * 100:5.1f}%")

## Step 4. 기본값 그대로 같은 질문 3번

temperature 를 따로 주지 않으면 **기본값 1** 로 동작합니다.
같은 질문을 3번 보내서 답을 나란히 놓고 비교해 봅니다.

In [ ]:
question = "보안 담당자에게 오늘 하루를 응원하는 한 문장을 써 주세요."

for i in range(3):
    print(f"[{i + 1}]", ask(question))

## Step 5. `temperature=0` — 가장 유력한 후보만 고른다

temperature 를 0 으로 주면 모델이 **모험을 하지 않고** 매번 확률이 가장 높은 후보를 고릅니다.
Step 4 와 **같은 질문**으로 다시 3번 보내 봅니다.

| temperature | 동작 | 어울리는 일 |
|---|---|---|
| `0` | 가장 유력한 후보만 | 분류, 정보 추출, 채점처럼 **답이 하나여야** 하는 일 |
| `1` (기본값) | 적당히 섞어서 | 일반 대화 |
| `1.5` 이상 | 낮은 확률 후보도 자주 | 아이디어 내기, 창작 |

> 3개가 거의 같게 나와도 **글자 하나까지 같다는 보장은 없습니다.**
> 서버의 GPU 연산 순서 같은 사소한 차이로 가끔 달라질 수 있습니다.

In [ ]:
for i in range(3):
    print(f"[{i + 1}]", ask(question, temperature=0))

## Step 6. `temperature=1.5` — 모험을 많이 한다

이번엔 반대로 temperature 를 크게 올립니다. 허용 범위는 `0 ~ 2` 입니다.
낮은 확률의 후보도 자주 뽑히므로 **표현이 다양해지지만, 엉뚱해지기도 합니다.**

> 2 에 가깝게 올려 보면 문장이 아예 무너지는 것도 볼 수 있습니다.
> "창의적"과 "엉망"은 생각보다 가까이 있습니다.

In [ ]:
for i in range(3):
    print(f"[{i + 1}]", ask(question, temperature=1.5))

## Step 7. `n=3` — 한 번 호출로 답 3개 받기

`1_1_openai` 의 Step 5 에서 `choices` 가 **목록**인 이유를 설명했습니다.
`n=3` 을 주면 같은 질문에 대한 답을 한 번에 3개 받고, `choices` 의 길이가 3 이 됩니다.

```
response
  ├ choices[0].message.content   <- 답 1
  ├ choices[1].message.content   <- 답 2
  └ choices[2].message.content   <- 답 3
```

Step 4 처럼 3번 부르는 것과 결과는 비슷하지만, **보내는 질문(입력 토큰)을 한 번만 내므로** 더 쌉니다.
여러 후보 중 하나를 고르게 하고 싶을 때 씁니다.

In [ ]:
response = openai.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": question}],
    max_tokens=60,
    n=3,
)

for choice in response.choices:
    print(f"[{choice.index + 1}]", choice.message.content)

## Step 8. `seed` — "가능한 한" 같은 답을 달라

temperature 가 0 이 아니어도, **같은 `seed` 값을 주면** OpenAI 가 최대한 같은 결과를 내려고 시도합니다.
게임에서 맵 생성 seed 를 같게 주면 같은 맵이 나오는 것과 비슷한 생각입니다.

```python
openai.chat.completions.create(..., seed=42)
```

### `system_fingerprint` 도 같이 보는 이유

응답의 `system_fingerprint` 는 **답을 만든 서버 설정의 지문**입니다.
같은 seed 를 줬는데도 답이 다르다면 이 값부터 비교해 보세요.
지문이 다르면 OpenAI 쪽 설정이 바뀐 것이므로, 같은 답을 기대할 수 없습니다.

> `seed` 는 **노력해 주겠다는 약속이지 보장이 아닙니다.**
> 모델에 따라 `system_fingerprint` 가 `None` 으로 나오기도 합니다.

In [ ]:
for i in range(3):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        max_tokens=60,
        temperature=0.7,
        seed=42,
    )
    print(f"[{i + 1}]", response.choices[0].message.content)
    print("    fingerprint:", response.system_fingerprint)

## Step 9. 추론 모델은 temperature 를 바꿀 수 없다

`1_1_openai` 의 Step 5 에서 쓴 `gpt-5-nano` 는 답하기 전에 스스로 생각하는 **추론(reasoning) 모델**입니다.
이런 모델은 temperature 를 기본값으로 고정해 두어서, 다른 값을 주면 **오류가 납니다.**

일부러 오류를 내 보고 메시지를 읽어 봅니다.
`try / except` 로 감쌌으므로 오류가 나도 노트북이 멈추지 않습니다.

> **오류 메시지를 읽는 습관**: API 오류는 대부분 무엇이 잘못됐는지 메시지에 그대로 적혀 있습니다.
> 여기서는 `temperature` 라는 단어와 "지원하지 않는다(unsupported)"는 말이 보일 것입니다.

In [ ]:
from openai import BadRequestError

try:
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": question}],
        temperature=0,
    )
    print("호출 성공:", response.choices[0].message.content)
except BadRequestError as e:
    print("오류가 났습니다:")
    print(e.message)

## Step 10. 100% 같은 답 — 답을 저장해 두고 다시 쓴다

지금까지 본 방법은 모두 **"거의 같게"** 까지입니다.
정말로 글자 하나까지 같아야 한다면 방법은 하나입니다. **처음 받은 답을 저장해 두고, 같은 질문이 오면 API 를 다시 부르지 않는 것.**

```
질문이 들어온다
   ├ 저장된 답이 있다  ->  저장된 답을 그대로 돌려준다   (API 호출 없음, 무료)
   └ 저장된 답이 없다  ->  API 를 부르고, 답을 파일에 저장한 뒤 돌려준다
```

이런 저장소를 **캐시(cache)** 라고 합니다. 같은 답을 보장할 뿐 아니라 **요금과 시간도 아껴 줍니다.**

| | 내용 |
|---|---|
| 저장 파일 | `answers_cache.json` (이 노트북과 같은 폴더) |
| 키 | 질문 문장 그대로 |
| 값 | 처음 받은 답 |

> 질문이 **한 글자라도 다르면** 다른 질문으로 취급해 API 를 새로 부릅니다.

In [ ]:
import json

CACHE_FILE = "answers_cache.json"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, encoding="utf-8") as f:
        cache = json.load(f)
else:
    cache = {}

def ask_cached(question):
    """저장된 답이 있으면 그대로, 없으면 API 를 불러 저장한 뒤 돌려줍니다."""
    if question in cache:
        print("(저장된 답 사용)")
        return cache[question]

    print("(API 호출)")
    answer = ask(question, temperature=0)
    cache[question] = answer
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)
    return answer

같은 질문을 3번 보냅니다. **첫 번째만 API 를 부르고**, 나머지는 저장된 답을 씁니다.
셀을 다시 실행하거나 노트북을 껐다 켜도 파일에 남아 있으므로 계속 같은 답이 나옵니다.

In [ ]:
for i in range(3):
    print(f"[{i + 1}]", ask_cached(question))

---

## 정리

```
답이 달라지는 이유: 모델은 다음 단어를 확률로 고른다
  -> temperature 로 얼마나 모험할지 정한다
  -> 그래도 100% 같아야 한다면 답을 저장해 두고 다시 쓴다
```

### 꼭 기억할 것

| 목표 | 방법 | 보장 |
|---|---|---|
| 다양한 답 | `temperature` 를 올린다 (`1 ~ 1.5`) | - |
| 거의 같은 답 | `temperature=0` + `seed` | 거의 같음 |
| 완전히 같은 답 | 처음 받은 답을 캐시에 저장 | 100% |
| 여러 후보 한 번에 | `n=3` | - |
| 추론 모델 (`gpt-5` 계열) | temperature 를 바꿀 수 없다 | - |

### 직접 해 보기

1. Step 3 의 질문을 "인공지능을 한 단어로 표현하면?" 으로 바꿔 보세요. 후보 확률이 한쪽에 몰리나요, 고르게 퍼지나요?
2. Step 6 의 temperature 를 `2.0` 으로 올려 보세요. 문장이 어떻게 무너지나요?
3. Step 5 를 여러 번 다시 실행해 보세요. temperature 0 인데도 답이 달라지는 경우가 있나요?
4. `ask_cached()` 로 질문 끝에 물음표만 하나 더 붙여 보내 보세요. 저장된 답을 쓰나요, API 를 새로 부르나요?
5. `answers_cache.json` 파일을 열어 내용을 확인하고, 지운 뒤 Step 10 을 다시 실행해 보세요.